In [2]:
# Week 4 - Day 1 (Monday): Setup + load cleaned dataset
# Step 1: Clone repo
!git clone https://github.com/mustafayubk/SOSC314_Project.git

# Step 2: Move into repo folder
import os
os.chdir("SOSC314_Project")

# Step 3: Confirm structure
!ls


Cloning into 'SOSC314_Project'...
remote: Enumerating objects: 190, done.
remote: Counting objects: 100% (190/190), done.
remote: Compressing objects: 100% (180/180), done.
remote: Total 190 (delta 96), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (190/190), 5.44 MiB | 7.71 MiB/s, done.
Resolving deltas: 100% (96/96), done.
 config   docs	      README.md  'Week 2_Figure.png'
 data	  notebooks   scripts	  Week_3_Figure.png


In [3]:
import pandas as pd

# Load cleaned comments (Week 3 output)
comments = pd.read_csv("data/processed/comments_clean_week3.csv")

# Load video metadata
videos = pd.read_csv("data/video_list.csv")

videos_meta = videos[["video_id", "category", "year", "time_bin"]]

df = comments.merge(videos_meta, on="video_id", how="left")

print("Comments rows:", len(df))
print("Unique videos:", df["video_id"].nunique())
print("Missing category rows:", df["category"].isna().sum())

df.head()


Comments rows: 22291
Unique videos: 45
Missing category rows: 0


,video_id,comment_id,text,like_count,published_at,comment_length,category,year,time_bin
0,iG9CE55wbtY,UgwVLE5mF37k2yEvg9p4AaABAg,2 decades still relevant!,0,2026-01-22T11:29:53Z,25,TED-style,2006,2000-2010
1,iG9CE55wbtY,UgzYwYBO7ip2UL8v9fp4AaABAg,its 2026 we have Elon musk.,0,2026-01-20T18:55:38Z,27,TED-style,2006,2000-2010
2,iG9CE55wbtY,UgwMS2_Mr8j1Cpu-YsF4AaABAg,Sarah was watching this TED Talk fuming,0,2026-01-20T17:11:00Z,39,TED-style,2006,2000-2010
3,iG9CE55wbtY,UgxVRI7jie5w09hRWTp4AaABAg,JEFF BESOS at 15:00 ?,0,2026-01-19T23:01:56Z,21,TED-style,2006,2000-2010
4,iG9CE55wbtY,UgwdC5qgXmDOkNhzYqp4AaABAg,I love how he presents and speaks. Very genuin...,0,2026-01-19T01:44:49Z,94,TED-style,2006,2000-2010


In [4]:
import os

os.makedirs("data/processed/week4", exist_ok=True)

summary = (
    df.groupby("category")
      .agg(
          n_comments=("comment_id", "count"),
          n_videos=("video_id", "nunique"),
          avg_comment_len=("comment_length", "mean")
      )
      .reset_index()
)

print("Summary by genre:")
display(summary)

df.to_csv(
    "data/processed/week4/comments_week4_ready.csv",
    index=False
)

print("Saved: data/processed/week4/comments_week4_ready.csv")


Summary by genre:


,category,n_comments,n_videos,avg_comment_len
0,Personal opinion,7498,15,117.825153
1,Short film,7493,15,77.432137
2,TED-style,7300,15,119.426712


Saved: data/processed/week4/comments_week4_ready.csv


In [5]:
# STEP 1: Install and import VADER sentiment tool
!pip install vaderSentiment

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

# STEP 2: Load cleaned comments from Week 3
comments = pd.read_csv("data/processed/comments_clean_week3.csv")

print("Number of comments loaded:", len(comments))

# STEP 3: Initialize the sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# STEP 4: Compute sentiment score for each comment
comments["sentiment_vader"] = comments["text"].astype(str).apply(
    lambda x: analyzer.polarity_scores(x)["compound"]
)

# STEP 5: Quick sanity check
comments[["text", "sentiment_vader"]].head()


Number of comments loaded: 22291


,text,sentiment_vader
0,2 decades still relevant!,0.0000
1,its 2026 we have Elon musk.,0.0000
2,Sarah was watching this TED Talk fuming,-0.5719
3,JEFF BESOS at 15:00 ?,0.0000
4,I love how he presents and speaks. Very genuin...,0.9422


In [6]:
# STEP 6: Save dataset with sentiment scores
import os

os.makedirs("data/processed/week4", exist_ok=True)

comments.to_csv(
    "data/processed/week4/comments_with_vader_sentiment.csv",
    index=False
)

print("Saved file: data/processed/week4/comments_with_vader_sentiment.csv")


Saved file: data/processed/week4/comments_with_vader_sentiment.csv
